# Workshop 1 — FITS files and astronomical plotting

## From a FITS file to a figure

FITS stands for Flexible Image Transport System. It is an astronomical data format that is widely used to share numerical data, usually images but often tabular data as well. A FITS image contains a numerical array **and** metadata describing what the array represents and how its pixels map onto the sky.

In this workshop you will learn how to:

- open and inspect FITS files with `astropy.io.fits`
- understand HDUs, headers and data arrays
- inspect image data numerically with NumPy
- display images with Matplotlib
- choose sensible image scaling
- use a FITS World Coordinate System (WCS)
- make an astronomical plot with celestial coordinates, contours and a colour bar
- save a figure for use elsewhere

<figure style="text-align: center;">
    <img
      src="images/SgrAstar.png"
      alt="Example of a plotted FITS image"
      width="700"
    >
    <figcaption>
        Figure 1: Observations of molecular clouds toward the Galactic Centre <a href="https://ui.adsabs.harvard.edu/abs/2019MNRAS.485.2457H/abstract" target="_blank"> (Henshaw et al. 2019) </a>
    </figcaption>
</figure>

### Files used

This workshop uses two FITS images stored in the top-level `data/` folder:

- `data/656nmos.fits`
- `data/mystery.fits`

The workshop notebooks sit at the project top level, alongside the `data/` folder.

## 1. Import the packages we need

We will use NumPy for arrays, Matplotlib for figures, and Astropy for FITS, WCS, and image stretches. The imports are intentionally small enough to reuse in a research notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch

## 2. Choose the FITS file

The notebooks are at the project top level and the FITS files are in `data/`. A direct relative path therefore keeps the scientific workflow easy to see.

In [ ]:
fits_file = "data/656nmos.fits"

## 3. Open a FITS file

A FITS file consists of one or more **Header/Data Units (HDUs)**.

Each HDU can contain:

1. a **header** containing metadata
2. a **data array** or table.

`fits.open()` returns an `HDUList`, which behaves rather like a Python list.

In [ ]:
hdul = fits.open(fits_file)
hdul

The `.info()` method gives a compact summary of the HDUs in the file.

In [ ]:
hdul.info()

Python uses **zero-based indexing**, so `hdul[0]` is the first HDU.

In [ ]:
primary_hdu = hdul[0]
primary_hdu

### Exercise 1

Use Python indexing to inspect the second HDU in the FITS file. Then use `.info()` above to identify whether it contains an image or a table.

In [ ]:
# Your code here
# hdul[...]

In [ ]:
# Solution
hdul[1]

## 4. FITS headers

The FITS header stores metadata as a collection of keyword/value pairs. Many headers are long, so in research you will normally inspect individual keywords rather than print the entire header.

In [ ]:
header = primary_hdu.header

print(header["NAXIS"])
print(header["NAXIS1"])
print(header["NAXIS2"])

Some particularly important keywords describe the celestial coordinate system. Their exact form varies between datasets, but `CTYPE1` and `CTYPE2` often identify the two sky-coordinate axes.

In [ ]:
for key in ["CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2"]:
    print(f"{key:6s} = {header.get(key)}")

You can also search a header using wildcard matching:

In [ ]:
header["*DATE*"]

### Exercise 2

Find the exposure time and target name in the FITS header. Hint: the relevant keywords in this dataset are `EXPTIME` and `TARGNAME`.

In [ ]:
# Your code here

In [ ]:
# Solution
print("Exposure time:", header["EXPTIME"], "s")
print("Target:", header["TARGNAME"])

## 5. The image is a NumPy array

The numerical image is stored in the `.data` attribute of the HDU.

In [ ]:
data = primary_hdu.data

print(type(data))
print(data.shape)
print(data.dtype)

For numerical analysis it is often useful to work with floating-point values explicitly.

In [ ]:
data = data.astype(float)

A two-dimensional image is indexed as:

```python
data[y, x]
```

The first index therefore refers to the row and the second to the column.

In [ ]:
ny, nx = data.shape

print("Image dimensions:", nx, "x", ny)
print("Central pixel:", data[ny // 2, nx // 2])

### Slicing an image

Array slicing lets us extract a subsection without changing the original image.

In [ ]:
cutout = data[600:1000, 600:1000]

print(cutout.shape)

## 6. Inspect the image numerically

Before plotting data, it is good practice to inspect it. `np.nan...` functions ignore any `NaN` (not-a-number) pixels.

In [ ]:
print(f"Minimum: {np.nanmin(data):.3f}")
print(f"Maximum: {np.nanmax(data):.3f}")
print(f"Mean:    {np.nanmean(data):.3f}")
print(f"Median:  {np.nanmedian(data):.3f}")
print(f"Std dev: {np.nanstd(data):.3f}")
print(f"NaNs:    {np.isnan(data).sum()}")

The minimum and maximum are not necessarily good choices for plotting limits. A small number of extreme pixels can dominate the dynamic range.

Percentiles are often more useful.

In [ ]:
p1, p50, p99 = np.nanpercentile(data, [1, 50, 99])

print("1st percentile: ", p1)
print("Median:         ", p50)
print("99th percentile:", p99)

### Exercise 3

Calculate the 5th and 99.5th percentiles of the image.

In [ ]:
# Your code here

In [ ]:
# Solution
p5, p995 = np.nanpercentile(data, [5, 99.5])
print(p5, p995)

## 7. Make the simplest possible image

Matplotlib's `imshow()` displays a two-dimensional array.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.imshow(data)

plt.show()

This is technically a plot, but it is not yet a particularly useful astronomical figure.

Notice several things:

- the axes show **pixel numbers**, not celestial coordinates
- the default scaling may hide faint structure
- no physical quantity is given for the colour scale
- the image origin may not be where you expect.

We will improve these one at a time.

## 8. Control the image display

By default, Matplotlib uses `origin="upper"`, which places index `[0, 0]` at the top-left (standard matrix and image convention) with the y-axis pointing downward. A common first step is to set `origin="lower"` and choose a colour map and display limits.

In [ ]:
vmin, vmax = np.nanpercentile(data, [5, 99.5])

fig, ax = plt.subplots(figsize=(7, 7))

im = ax.imshow(
    data,
    origin="lower",
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
)

ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
ax.set_title("F656N image")

fig.colorbar(im, ax=ax, label="Pixel value")

plt.show()

### Why scaling matters

An astronomical image may contain emission spanning a large range of intensities. The choice of display stretch changes which structures are visible.

This does **not** alter the underlying data. It alters only the mapping between numerical values and displayed colours.

Astropy provides several useful image-stretching tools. Here we combine a percentile interval with an inverse-hyperbolic-sine (`asinh`) stretch.

In [ ]:
interval = PercentileInterval(99.5)
vmin, vmax = interval.get_limits(data)

norm = ImageNormalize(
    vmin=vmin,
    vmax=vmax,
    stretch=AsinhStretch(),
)

fig, ax = plt.subplots(figsize=(7, 7))

im = ax.imshow(
    data,
    origin="lower",
    cmap="gray",
    norm=norm,
)

fig.colorbar(im, ax=ax, label="Pixel value")
ax.set_title("F656N image with asinh scaling")

plt.show()

### Exercise 4

Make two plots of the same image:

1. one using a linear display between the 1st and 99th percentiles
2. one using the `asinh` normalization above.

Which structures are easier to see in each?

In [ ]:
# Your code here

## 9. From pixels to sky coordinates: WCS

The image header contains enough information to describe how image pixels map onto coordinates on the sky. Astropy represents this information using a `WCS` object.

In [ ]:
wcs = WCS(header)
wcs

Astropy may report a `FITSFixedWarning` here because it normalises the historical `DATE-OBS` value in this file. This metadata correction is expected and does not invalidate the image WCS.

Matplotlib can use this WCS directly when creating an axis.

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection=wcs)

im = ax.imshow(
    data,
    origin="lower",
    cmap="gray",
    norm=norm,
)

ax.set_xlabel("Right Ascension")
ax.set_ylabel("Declination")
ax.set_title("M16 — HST F656N")
ax.coords.grid(color="white", alpha=0.4, linestyle=":")

fig.colorbar(im, ax=ax, label="Pixel value", pad=0.05)

plt.show()

The plotted array has not changed. The WCS is being used to transform the **axes** from pixel coordinates into celestial coordinates.

The celestial tick marks are slanted because this detector's pixel grid is rotated relative to the RA–Dec grid. WCSAxes aligns each tick with the local celestial coordinate line, so the ticks meet the rectangular image boundary at an angle. The dotted grid makes this geometry visible; it is a correct representation of the FITS WCS, not a plotting error.

This is an important general principle:

> **The data array contains the measured values; the header tells us how those values relate to the physical or celestial world.**

## 10. Add contours

Contours are useful for showing surface-brightness structure and are commonly used when comparing astronomical images.

In [ ]:
levels = np.nanpercentile(data, [99.9])

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection=wcs)

im = ax.imshow(
    data,
    origin="lower",
    cmap="gray",
    norm=norm,
)

ax.contour(
    data,
    levels=levels,
    colors="red",
    linewidths=0.8,
    alpha=0.8,
)

ax.set_xlabel("Right Ascension")
ax.set_ylabel("Declination")
ax.set_title("M16 — HST F656N")

fig.colorbar(im, ax=ax, label="Pixel value", pad=0.05)

plt.show()

### A note on contour levels

There is no universally correct set of contour levels. In research they are often tied to a physically meaningful quantity, for example multiples of the image RMS noise. Percentile-based contours are used here simply to show the brightest regions in the image (associated with the field stars).

## 11. Produce a final figure

A good scientific figure should communicate what is being shown without requiring the reader to inspect your code.

At minimum, consider:

- sensible axis labels
- physical/celestial coordinates where appropriate
- a useful colour scale
- units
- a concise title or caption
- readable text sizes
- display limits chosen for a scientific reason.

For the final layout, `make_axes_locatable` creates a dedicated colour-bar axis immediately beside the image. Because this new axis is tied to the image axis, the colour bar has the same height as the displayed image. This uses page space efficiently and gives a clean, publication-quality result.

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection=wcs)

im = ax.imshow(
    data,
    origin="lower",
    cmap="gray",
    norm=norm,
)

ax.contour(
    data,
    levels=levels,
    colors="white",
    linewidths=0.8,
)

ax.coords[0].set_axislabel("Right Ascension")
ax.coords[1].set_axislabel("Declination")
ax.coords[0].set_ticklabel(exclude_overlapping=True)

ax.set_title("M16 — HST/WFPC2 F656N")

divider = make_axes_locatable(ax)
cax = divider.append_axes(
    "right", size="4%", pad=0.08, axes_class=plt.Axes
)
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Pixel value")

fig.tight_layout()

plt.show()

## 12. Save the figure

For a paper or report, vector formats such as PDF are useful for line art and text. PNG is convenient for slides and webpages.

In [ ]:
output_file = "workshop1_figure_S12.png"
fig.savefig(output_file, dpi=200, bbox_inches="tight")
print(f"Saved {output_file}")

# Final Task: Mystery Image

Use `data/mystery.fits` and create a publication-quality figure. Begin by inspecting the file. The image dimensions, numerical range and header information may differ from the guided examples above.

Your figure should:

1. open the FITS file and inspect its HDUs
2. extract the image data and header
3. report the image shape, median and standard deviation
4. choose and justify sensible display limits or a stretch
5. use the WCS to display Right Ascension and Declination
6. include a colour bar
7. add at least three contour levels
8. include an informative title
9. save the result as a PNG or PDF.

In [ ]:
# Final exercise workspace

# 1. Choose a file
# 2. Open it
# 3. Inspect the data numerically
# 4. Build a WCS
# 5. Make and save your figure

# Assessment

You should be prepared to discuss your code for the final task during the viva for this lab. Bring a copy of this notebook with your completed analysis included.